In [1]:
import itertools
import pandas as pd
import geopandas as gpd
from pathlib import Path
from libpysal import graph
from esda.moran import Moran
import numpy as np

In [2]:
shapes = ['square','pent','hex']
sizes = [25,100,400]
n_runs = 10
rhos = np.arange(-0.9, 1.0, 0.1)

In [5]:
def compute_truth_autocorrelation_synthetic(shape, size, rho, n_run, gdf):

    graph_path = f"graphs/{shape}/size_{size}/g_true.parquet"
    g = graph.read_parquet(graph_path)
    wm = g.to_W()
    wm.transform = 'r'

    node_positions = list(wm.id_order)
    col = f"rho_{rho:.1f}_run_{n_run}"
    y = gdf.iloc[node_positions][col].values

    mi = Moran(y, wm)
    global_result = {
        "shape": shape, "size": size,
        "rho": rho, "data_run": n_run,
        "moran_i": mi.I, "moran_p": mi.p_sim, "moran_z": mi.z_sim,
    }

    return global_result

In [8]:
for shape, size in itertools.product(shapes, sizes):
    src_path = f"data/autocorrelation/gdf_{shape}_{size}.parquet"
    gdf = gpd.read_parquet(src_path).reset_index(drop=True)

    truth_global_results = []

    for rho in rhos:
        for n_run in range(n_runs):
            global_result = compute_truth_autocorrelation_synthetic(
                shape, size, rho, n_run, gdf
            )
            truth_global_results.append(global_result)

    # save globals
    global_dir = Path(f"results/autocorrelation/synthetic/truth/global")
    pd.DataFrame(truth_global_results).to_parquet(
        global_dir / f"{shape}_{size}.parquet"
    )
    print(f"Done: {shape} size_{size}")

Done: square size_25
Done: square size_100
Done: square size_400
Done: pent size_25
Done: pent size_100
Done: pent size_400
Done: hex size_25
Done: hex size_100
Done: hex size_400
